# Seasonal Product Demand Analysis

**Project type:** Exploratory Data Analysis and Visualization (EDAV)  
**Dataset:** UCI Online Retail II  
**Main question:** Which products show a repeated peak month in comparable January–November windows of 2010 and 2011?

This notebook inspects and cleans retail transaction data, compares monthly quantity and revenue, evaluates recurring product-level demand peaks, and tests how the candidate list changes under alternative screening thresholds. The workflow is organized as five course questions (Q1–Q5), with data validation and export checks at the end.

**Important interpretation:** the 292 products are exploratory recurring-peak candidates, not statistically proven long-term seasonal products. Only two comparable annual windows are used. December is excluded from the primary comparison because the recorded December periods are incomplete.

In [ ]:
from pathlib import Path
import pandas as pd
import urllib.request

# Detect Google Colab, while keeping the same notebook usable in local Jupyter.
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None
    IN_COLAB = False

if IN_COLAB:
    PROJECT_DIR = Path("/content")
else:
    current_dir = Path.cwd()
    if (current_dir / "README.md").exists():
        PROJECT_DIR = current_dir
    elif (current_dir.parent / "README.md").exists():
        PROJECT_DIR = current_dir.parent
    else:
        PROJECT_DIR = current_dir

DATA_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx"
file_path = DATA_DIR / "online_retail_II.xlsx"

if not file_path.exists():
    print("Downloading the UCI Online Retail II workbook (~43.5 MB)...")
    try:
        urllib.request.urlretrieve(DATASET_URL, file_path)
    except Exception as exc:
        raise RuntimeError(
            "Dataset download failed. Download Online Retail II from "
            "https://archive.ics.uci.edu/dataset/502/online+retail+ii and "
            f"place online_retail_II.xlsx at: {file_path}"
        ) from exc
else:
    print("Using the existing dataset file:", file_path)

print("Dataset file:", file_path)
print("File size (MB):", round(file_path.stat().st_size / (1024**2), 2))
print("Output directory:", OUTPUT_DIR)

excel_file = pd.ExcelFile(file_path)
print("Available sheets:", excel_file.sheet_names)

In [ ]:
# Load both sheets from the Excel workbook
sheet_1 = pd.read_excel(file_path, sheet_name="Year 2009-2010")
sheet_2 = pd.read_excel(file_path, sheet_name="Year 2010-2011")

# Combine both years into one DataFrame
df = pd.concat([sheet_1, sheet_2], ignore_index=True)

# Inspect the dataset
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

## Setup: Initial Data Quality Audit

Inspect the raw data types, missing values, duplicate rows, date coverage, quantity and price anomalies, cancellation-coded invoice records, and dataset cardinalities before cleaning.

In [ ]:
# 1. Check column data types
print("DATA TYPES")
print(df.dtypes)

# 2. Check missing values
print("\nMISSING VALUES")
missing = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2)
})
display(missing)

# 3. Check duplicate rows
print("\nDUPLICATE ROWS")
print("Duplicate count:", df.duplicated().sum())

# 4. Check transaction dates
print("\nDATE RANGE")
print("First transaction:", df["InvoiceDate"].min())
print("Last transaction:", df["InvoiceDate"].max())

# 5. Check quantities and prices
print("\nQUANTITY AND PRICE")
print("Negative quantities:", (df["Quantity"] < 0).sum())
print("Zero quantities:", (df["Quantity"] == 0).sum())
print("Negative prices:", (df["Price"] < 0).sum())
print("Zero prices:", (df["Price"] == 0).sum())

# 6. Check cancelled invoices
print("\nCANCELLED INVOICES")
cancelled = df["Invoice"].astype(str).str.startswith("C")
print("Rows with cancellation invoice codes:", cancelled.sum())

# 7. Count unique products, invoices and countries
print("\nDATASET OVERVIEW")
print("Unique products:", df["StockCode"].nunique())
print("Unique invoices:", df["Invoice"].nunique())
print("Unique countries:", df["Country"].nunique())

## Setup: Data Cleaning and Preparation

The following cleaning steps are applied sequentially: exact duplicate removal, cancellation-coded invoice removal, non-positive quantity removal, negative-price removal, and removal of missing product descriptions. Customer IDs are retained when missing, and zero-price records that remain after cleaning are kept for quantity analysis.

In [ ]:
# Keep the original dataset unchanged
df_clean = df.copy()

original_rows = len(df_clean)

# Step 1: Remove exact duplicate rows
duplicates_removed = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates().copy()

# Step 2: Remove cancelled invoices
cancelled_mask = (
    df_clean["Invoice"]
    .astype(str)
    .str.strip()
    .str.startswith("C")
)

cancelled_removed = cancelled_mask.sum()
df_clean = df_clean.loc[~cancelled_mask].copy()

# Step 3: Remove non-positive quantities
invalid_quantity_mask = df_clean["Quantity"] <= 0
quantity_removed = invalid_quantity_mask.sum()
df_clean = df_clean.loc[~invalid_quantity_mask].copy()

# Step 4: Remove negative prices
negative_price_mask = df_clean["Price"] < 0
negative_prices_removed = negative_price_mask.sum()
df_clean = df_clean.loc[~negative_price_mask].copy()

# Step 5: Remove rows with missing product descriptions
missing_description_mask = df_clean["Description"].isna()
missing_descriptions_removed = missing_description_mask.sum()
df_clean = df_clean.loc[~missing_description_mask].copy()

# Step 6: Calculate transaction revenue
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["Price"]

# Step 7: Display cleaning summary
print("DATA CLEANING SUMMARY")
print("-" * 40)
print("Original rows:", original_rows)
print("Exact duplicates removed:", duplicates_removed)
print("Cancelled invoice rows removed:", cancelled_removed)
print("Non-positive quantity rows removed:", quantity_removed)
print("Negative price rows removed:", negative_prices_removed)
print("Missing descriptions removed:", missing_descriptions_removed)
print("Final cleaned rows:", len(df_clean))

print("\nCLEANED DATA PREVIEW")
display(df_clean.head())

print("\nRemaining missing values:")
display(df_clean.isnull().sum())

In [ ]:
# Setup: Create date features and the overall monthly demand summary

df_clean["Year"] = df_clean["InvoiceDate"].dt.year
df_clean["Month"] = df_clean["InvoiceDate"].dt.month
df_clean["Month_Name"] = df_clean["InvoiceDate"].dt.month_name()
df_clean["Date"] = df_clean["InvoiceDate"].dt.normalize()

daily = df_clean.groupby("Date").agg(
    Daily_Units=("Quantity", "sum"),
    Daily_Revenue=("Revenue", "sum"),
    Daily_Orders=("Invoice", "nunique")
)

# Fill missing dates for the daily series; December months are not used
# in the primary year-over-year comparison because coverage is partial.
daily = daily.asfreq("D", fill_value=0)
daily["Year"] = daily.index.year
daily["Month"] = daily.index.month
daily["YearMonth"] = daily.index.to_period("M").astype(str)

monthly = daily.groupby(["Year", "Month", "YearMonth"]).agg(
    Total_Units=("Daily_Units", "sum"),
    Total_Revenue=("Daily_Revenue", "sum"),
    Total_Orders=("Daily_Orders", "sum"),
    Average_Units_Per_Day=("Daily_Units", "mean"),
    Active_Days=("Daily_Units", lambda x: (x > 0).sum()),
    Calendar_Days=("Daily_Units", "size")
).reset_index()

print("MONTHLY DEMAND SUMMARY")
display(monthly)
print("\nHIGHEST MONTHLY UNIT TOTALS (RAW MONTHLY VIEW; DECEMBER COVERAGE IS PARTIAL)")
display(
    monthly.sort_values("Total_Units", ascending=False)
    [["YearMonth", "Total_Units", "Total_Revenue"]].head(10)
)

In [ ]:
# Verify recorded transaction coverage for each December
import pandas as pd

for month in ["2009-12", "2010-12", "2011-12"]:
    period = pd.Period(month, freq="M")

    records = df.loc[
        df["InvoiceDate"].dt.to_period("M") == period
    ]

    print(f"\n{month}")
    print("First recorded transaction:", records["InvoiceDate"].min())
    print("Last recorded transaction:", records["InvoiceDate"].max())
    print("Distinct transaction dates:", records["InvoiceDate"].dt.date.nunique())
    print("Transaction rows:", len(records))

In [ ]:
# Audit product codes that differ only by capitalization

audit_df = df_clean.assign(
    Original_Code=df_clean["StockCode"].astype(str).str.strip(),
    Normalized_Code=df_clean["StockCode"].astype(str).str.strip().str.upper(),
    Description_Normalized=(
        df_clean["Description"].astype("string").str.strip().str.upper()
        .str.replace(r"\s+", " ", regex=True)
    )
)

code_audit = audit_df.groupby("Normalized_Code").agg(
    Original_Codes=("Original_Code", lambda s: sorted(s.unique().tolist())),
    Variant_Count=("Original_Code", "nunique"),
    Description_Count=("Description_Normalized", "nunique"),
    Descriptions=("Description_Normalized", lambda s: sorted(s.dropna().astype(str).unique().tolist()))
).reset_index()

variants = code_audit[code_audit["Variant_Count"] > 1].copy()
safe_codes = code_audit.loc[
    (code_audit["Variant_Count"] > 1) &
    (code_audit["Description_Count"] == 1),
    "Normalized_Code"
].tolist()
conflicting_codes = code_audit.loc[
    (code_audit["Variant_Count"] > 1) &
    (code_audit["Description_Count"] > 1),
    "Normalized_Code"
].tolist()

print("Stock-code groups with capitalization variants:", len(variants))
print("Groups eligible for consolidation under the matching-description rule:", len(safe_codes))
print("Groups retained separately because descriptions conflict:", len(conflicting_codes))
print("\nExamples of conflicting-description groups (first 15):")
display(
    variants[variants["Number_of_Descriptions"] > 1]
    [["Normalized_Code", "Original_Codes", "Number_of_Descriptions", "Descriptions"]]
    .head(15)
)

In [ ]:
# Prepare consistent product keys without overwriting original stock codes

work = df_clean.copy()
work["Original_Code"] = work["StockCode"].astype(str).str.strip()
work["Normalized_Code"] = work["Original_Code"].str.upper()
work["Description_Normalized"] = (
    work["Description"].astype("string").str.strip().str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

work["ProductKey"] = work["Original_Code"]
safe_mask = work["Normalized_Code"].isin(safe_codes)
work.loc[safe_mask, "ProductKey"] = work.loc[safe_mask, "Normalized_Code"]

# Explicitly derive the year/month fields used in all downstream analyses.
work["Year"] = work["InvoiceDate"].dt.year
work["Month"] = work["InvoiceDate"].dt.month

print("Rows prepared for product analysis:", len(work))
print("Product keys after selective capitalization harmonization:", work["ProductKey"].nunique())
print("Original StockCode values are preserved in the dataset.")
print("Only case variants with a single normalized description are consolidated.")

## Q1. NumPy — Statistical Analysis of Monthly Retail Demand [CO1, BL3]

Question: How does monthly unit demand vary across January–November 2010 and the same months in 2011?

Concepts: Convert monthly totals to NumPy arrays and calculate mean, median, minimum, maximum and standard deviation. These statistics summarize the 11 comparable monthly totals for each year; they are not transaction-level statistics.

In [ ]:
# Q1: NumPy — Monthly retail-demand statistics [CO1, BL3]

import numpy as np
import pandas as pd

q1_data = df_clean[
    df_clean["Year"].isin([2010, 2011]) &
    df_clean["Month"].between(1, 11)
].copy()

# One value per month: total units sold in that month.
monthly_units_q1 = (
    q1_data.groupby(["Year", "Month"])["Quantity"]
    .sum()
    .unstack("Year")
    .reindex(index=range(1, 12), columns=[2010, 2011], fill_value=0)
)

month_names_q1 = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov"
]
monthly_units_q1.index = month_names_q1

summary_rows = []
for year in [2010, 2011]:
    values = monthly_units_q1[year].to_numpy(dtype=np.float64)
    summary_rows.append({
        "Year": year,
        "Mean monthly units": round(float(np.mean(values)), 2),
        "Median monthly units": round(float(np.median(values)), 2),
        "Minimum monthly units": int(np.min(values)),
        "Maximum monthly units": int(np.max(values)),
        "Standard deviation": round(float(np.std(values)), 2)
    })

q1_summary = pd.DataFrame(summary_rows)
print("NUMPY SUMMARY OF THE 11 COMPARABLE MONTHLY UNIT TOTALS")
display(q1_summary)
print("\nMonthly totals used in the statistics:")
display(monthly_units_q1)

## Q2. Pandas — Monthly Sales Quantity and Revenue [CO2, BL3]

Question: Which months record the highest quantities and revenue, and how do these measures compare between 2010 and 2011?

Concepts: Date-based selection, `pivot_table`, aggregation and visual comparison. The primary heatmaps use January–November only.

In [ ]:
# Q2: Monthly Demand Heatmaps — Quantity and Revenue

import pandas as pd
import matplotlib.pyplot as plt

# Prepare the comparable January-November period
heatmap_df = df_clean.copy()

heatmap_df["Year"] = heatmap_df["InvoiceDate"].dt.year
heatmap_df["Month"] = heatmap_df["InvoiceDate"].dt.month

heatmap_df = heatmap_df[
    heatmap_df["Year"].isin([2010, 2011]) &
    heatmap_df["Month"].between(1, 11)
].copy()

month_labels = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov"
]

# Aggregate monthly quantity and revenue
monthly_heatmap = heatmap_df.pivot_table(
    index="Year",
    columns="Month",
    values=["Quantity", "Revenue"],
    aggfunc="sum",
    fill_value=0
)

units_table = monthly_heatmap["Quantity"].reindex(
    index=[2010, 2011], columns=range(1, 12), fill_value=0
)

revenue_table = monthly_heatmap["Revenue"].reindex(
    index=[2010, 2011], columns=range(1, 12), fill_value=0
)

# Create two heatmaps
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Heatmap A: Units sold
im1 = axes[0].imshow(
    units_table.values, aspect="auto", cmap="YlOrRd"
)
axes[0].set_title("Monthly Demand Heatmap — Total Units Sold")
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(["2010", "2011"])
axes[0].set_xticks(range(11))
axes[0].set_xticklabels(month_labels)
axes[0].set_xlabel("Month")

for i in range(2):
    for j in range(11):
        axes[0].text(
            j, i, f"{units_table.iloc[i, j]:,.0f}",
            ha="center", va="center", fontsize=8
        )

fig.colorbar(im1, ax=axes[0], label="Total units sold")

# Heatmap B: Revenue
revenue_thousands = revenue_table / 1000

im2 = axes[1].imshow(
    revenue_thousands.values, aspect="auto", cmap="YlGnBu"
)
axes[1].set_title("Monthly Demand Heatmap — Total Revenue")
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["2010", "2011"])
axes[1].set_xticks(range(11))
axes[1].set_xticklabels(month_labels)
axes[1].set_xlabel("Month")

for i in range(2):
    for j in range(11):
        axes[1].text(
            j, i, f"{revenue_thousands.iloc[i, j]:,.0f}",
            ha="center", va="center", fontsize=8
        )

fig.colorbar(
    im2, ax=axes[1],
    label="Revenue (thousands of recorded price units)"
)

fig.suptitle(
    "Retail Demand Comparison: January–November 2010 vs 2011",
    fontsize=14
)
plt.tight_layout()
plt.show()

# Save the data used to create the heatmaps
monthly_heatmap_export = (
    heatmap_df.groupby(["Year", "Month"])
    .agg(
        Total_Units=("Quantity", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Distinct_Invoices=("Invoice", "nunique")
    )
    .reset_index()
)

print("MONTHLY HEATMAP DATA")
display(monthly_heatmap_export)

In [ ]:
import matplotlib.pyplot as plt

# December observations are partial in 2009, 2010 and 2011; exclude all Decembers.
complete_months = monthly.loc[monthly["Month"] != 12].copy()

complete_months["Month_Date"] = pd.to_datetime(
    complete_months["YearMonth"] + "-01"
)

# CHART 1: Overall monthly demand trend
plt.figure(figsize=(12, 5))
plt.plot(
    complete_months["Month_Date"],
    complete_months["Total_Units"],
    marker="o"
)
plt.title("Monthly Product Demand Trend (Incomplete December Periods Excluded)")
plt.xlabel("Month")
plt.ylabel("Total Units Sold")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "monthly_product_demand_trend.png", dpi=200, bbox_inches="tight")
plt.show()

# CHART 2: Compare the same months in 2010 and 2011
comparison = monthly.loc[
    monthly["Year"].isin([2010, 2011]) &
    (monthly["Month"] <= 11)
]

month_comparison = comparison.pivot(
    index="Month",
    columns="Year",
    values="Total_Units"
)

month_names = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov"
]

plt.figure(figsize=(10, 5))
plt.plot(
    month_names,
    month_comparison.loc[1:11, 2010],
    marker="o",
    label="2010"
)
plt.plot(
    month_names,
    month_comparison.loc[1:11, 2011],
    marker="o",
    label="2011"
)
plt.title("Monthly Demand Comparison: 2010 vs 2011")
plt.xlabel("Month")
plt.ylabel("Total Units Sold")
plt.legend(title="Year")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "monthly_demand_year_over_year.png", dpi=200, bbox_inches="tight")
plt.show()

### Additional EDA: Investigating the March Demand Difference

This focused check inspects the products contributing the largest quantities in March 2010 and compares total March quantities with March 2011. It is exploratory and does not by itself establish why demand differed.

In [ ]:
# Investigate the March 2010 demand spike

march_2010 = df_clean[
    (df_clean["Year"] == 2010) &
    (df_clean["Month"] == 3)
]

march_products = (
    march_2010.groupby(["StockCode", "Description"])["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)

print("TOP 15 PRODUCTS BY QUANTITY — MARCH 2010")
display(march_products)

print("\nTotal units in March 2010:", march_2010["Quantity"].sum())
print("Total units in March 2011:", df_clean.loc[
    (df_clean["Year"] == 2011) &
    (df_clean["Month"] == 3),
    "Quantity"
].sum())

## Q3. Data Cleaning and Validation [CO3, BL4]

Question: Which data-quality issues could distort product-demand analysis, and how do the documented cleaning decisions affect the analysis-ready dataset?

The cleaning is performed earlier in the notebook so that all subsequent analyses use the same prepared data. The table below validates the row counts at each sequential step, compares missing values only for the original eight columns, and records zero-price rows retained. Negative and zero quantities are excluded from the primary positive-sales analysis; results therefore are not net sales after returns.

In [ ]:
# Q3: Data Cleaning Summary and Validation [CO3, BL4]

import pandas as pd

# Only compare the eight columns that existed in the original dataset.
original_columns = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country"
]

# Record row counts at each sequential cleaning stage.
steps = [
    ("Original dataset", 0),
    ("Exact duplicate removal", duplicates_removed),
    ("Cancelled invoice removal", cancelled_removed),
    ("Non-positive quantity removal", quantity_removed),
    ("Negative price removal", negative_prices_removed),
    ("Missing description removal", missing_descriptions_removed)
]

rows_before = original_rows
cleaning_records = []
for step_name, removed in steps:
    rows_after = rows_before - removed
    cleaning_records.append({
        "Cleaning Step": step_name,
        "Rows Before": rows_before,
        "Rows Removed": removed,
        "Rows After": rows_after
    })
    rows_before = rows_after

cleaning_summary = pd.DataFrame(cleaning_records)
print("DATA CLEANING SUMMARY — ROW COUNTS")
display(cleaning_summary)

missing_summary = pd.DataFrame({
    "Missing Before": df[original_columns].isna().sum(),
    "Missing After": df_clean[original_columns].isna().sum()
})
print("\nMISSING VALUES BEFORE AND AFTER CLEANING (ORIGINAL COLUMNS)")
display(missing_summary)

zero_prices_before = int((df["Price"] == 0).sum())
zero_prices_after = int((df_clean["Price"] == 0).sum())
print("\nZERO-PRICE AUDIT")
print("Zero-price rows before cleaning:", zero_prices_before)
print("Zero-price rows retained:", zero_prices_after)

expected_final_rows = original_rows - sum(removed for _, removed in steps)
print("\nFINAL VALIDATION")
print("Calculated final row count:", expected_final_rows)
print("Actual cleaned row count:", len(df_clean))
assert expected_final_rows == len(df_clean), "Cleaning counts do not reconcile."
assert len(df_clean) == 1_008_892, "Unexpected cleaned row count."
print("Validation passed: cleaning counts reconcile.")

## Q4. Product-Level Recurring Peaks [CO4, BL4]

Question: Which individual products show the same peak month in both comparison years, and how strong is the evidence?

Candidate screening rules:
1. At least 10 distinct invoices in each year.
2. Same peak month in both years.
3. Peak-to-average monthly quantity ratio of at least 1.5 in each year.
4. At least 10 distinct invoices in the peak month in both years.

Monthly-profile correlation is supporting evidence only. These rules are exploratory, and the resulting list should be described as recurring-peak candidates, not definitive long-term seasonality.

In [ ]:
import numpy as np
import pandas as pd

# Q4: Product-level recurring peak analysis [CO4, BL4]
# Primary analysis: January-November 2010 vs January-November 2011
primary_data = work[
    work["Year"].isin([2010, 2011]) &
    work["Month"].between(1, 11)
].copy()

print("Analysis period: January-November 2010 vs January-November 2011")
print("Rows:", len(primary_data))

# Aggregate units and distinct invoices by product, year, and month
product_month_primary = primary_data.groupby(
    ["ProductKey", "Year", "Month"]
).agg(
    Units=("Quantity", "sum"),
    Orders=("Invoice", "nunique")
).reset_index()

# Select the most frequently observed description for each product
labels_data = primary_data.assign(
    Description_Label=primary_data["Description"]
    .astype("string").str.strip()
)

product_names_primary = (
    labels_data.groupby(["ProductKey", "Description_Label"])
    .size()
    .rename("Frequency")
    .reset_index()
    .sort_values("Frequency", ascending=False)
    .drop_duplicates("ProductKey")
    .set_index("ProductKey")["Description_Label"]
)

# Ensure every product has columns for all 11 months in both years
expected = pd.MultiIndex.from_product(
    [[2010, 2011], range(1, 12)],
    names=["Year", "Month"]
)

units_primary = product_month_primary.pivot(
    index="ProductKey",
    columns=["Year", "Month"],
    values="Units"
).reindex(columns=expected).fillna(0)

orders_primary = product_month_primary.pivot(
    index="ProductKey",
    columns=["Year", "Month"],
    values="Orders"
).reindex(
    index=units_primary.index,
    columns=expected
).fillna(0)

u10_primary = units_primary.xs(2010, axis=1, level="Year")
u11_primary = units_primary.xs(2011, axis=1, level="Year")
o10_primary = orders_primary.xs(2010, axis=1, level="Year")
o11_primary = orders_primary.xs(2011, axis=1, level="Year")

# Peak month and peak-to-average ratio for each year
peak10_primary = u10_primary.idxmax(axis=1)
peak11_primary = u11_primary.idxmax(axis=1)

index10_primary = (
    u10_primary.max(axis=1) /
    u10_primary.mean(axis=1).replace(0, np.nan)
)

index11_primary = (
    u11_primary.max(axis=1) /
    u11_primary.mean(axis=1).replace(0, np.nan)
)

def peak_invoice_counts(monthly_orders, peak_months):
    values = monthly_orders.to_numpy()[
        np.arange(len(monthly_orders)),
        peak_months.to_numpy() - 1
    ]
    return pd.Series(values, index=monthly_orders.index)

# Compare the full 11-month demand profiles
correlations = {}

for key in units_primary.index:
    x = u10_primary.loc[key].to_numpy(dtype=float)
    y = u11_primary.loc[key].to_numpy(dtype=float)

    if np.std(x) > 0 and np.std(y) > 0:
        correlations[key] = np.corrcoef(x, y)[0, 1]
    else:
        correlations[key] = np.nan

profile_primary = pd.DataFrame(index=units_primary.index)
profile_primary["Description"] = product_names_primary.reindex(
    profile_primary.index
)
profile_primary["Peak_Month_2010"] = peak10_primary
profile_primary["Peak_Month_2011"] = peak11_primary
profile_primary["Peak_Index_2010"] = index10_primary.round(2)
profile_primary["Peak_Index_2011"] = index11_primary.round(2)
profile_primary["Invoices_2010"] = o10_primary.sum(axis=1).astype(int)
profile_primary["Invoices_2011"] = o11_primary.sum(axis=1).astype(int)

profile_primary["Peak_Invoices_2010"] = (
    peak_invoice_counts(o10_primary, peak10_primary).astype(int)
)
profile_primary["Peak_Invoices_2011"] = (
    peak_invoice_counts(o11_primary, peak11_primary).astype(int)
)

profile_primary["Profile_Correlation"] = pd.Series(correlations)
profile_primary["Same_Peak_Month"] = (
    profile_primary["Peak_Month_2010"] ==
    profile_primary["Peak_Month_2011"]
)
profile_primary["Repeat_Strength"] = profile_primary[
    ["Peak_Index_2010", "Peak_Index_2011"]
].min(axis=1)

month_names = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
    9: "Sep", 10: "Oct", 11: "Nov"
}
profile_primary["Peak_Month_Name"] = (
    profile_primary["Peak_Month_2010"].map(month_names)
)

# Screening rules: evidence of activity and recurring peaks
eligible_primary = profile_primary[
    (profile_primary["Invoices_2010"] >= 10) &
    (profile_primary["Invoices_2011"] >= 10)
].copy()

repeated_primary = eligible_primary[
    eligible_primary["Same_Peak_Month"] &
    (eligible_primary["Peak_Index_2010"] >= 1.5) &
    (eligible_primary["Peak_Index_2011"] >= 1.5)
].copy()

watchlist_primary = repeated_primary[
    (repeated_primary["Peak_Invoices_2010"] >= 10) &
    (repeated_primary["Peak_Invoices_2011"] >= 10)
].copy()

watchlist_primary = watchlist_primary.sort_values(
    ["Profile_Correlation", "Repeat_Strength"],
    ascending=False,
    na_position="last"
)

print("\nPRIMARY ANALYSIS COUNTS")
print("Eligible products:", len(eligible_primary))
print("Products with recurring peak months:", len(repeated_primary))
print("Candidates with at least 10 peak-month invoices in both years:",
      len(watchlist_primary))

print("\nTOP 20 CANDIDATES")
display(watchlist_primary[[
    "Description",
    "Peak_Month_Name",
    "Peak_Index_2010",
    "Peak_Index_2011",
    "Peak_Invoices_2010",
    "Peak_Invoices_2011",
    "Profile_Correlation"
]].head(20))

print("\nPEAK-MONTH DISTRIBUTION")
distribution_primary = (
    watchlist_primary["Peak_Month_Name"]
    .value_counts()
    .reindex(
        ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
         "Jul", "Aug", "Sep", "Oct", "Nov"],
        fill_value=0
    )
)
print(distribution_primary.to_string())

### Q4 Interpretation

The primary analysis reports the eligible-product count, the number with repeated peak months, and the final candidate count after peak-month invoice support. The main result from the current analysis is 2,358 eligible products and 292 recurring-peak candidates; November is the most frequent peak month, with 156 candidates.

### Representative Product Validation

Three readable examples are selected to illustrate recurring peaks in **March, October and November**. Their monthly units and distinct invoice activity are compared across both years. The January example is intentionally not used in this figure; the complete watchlist still includes candidates with peaks in every month from January through November.

In [ ]:
# Q4: Validate Representative Products Across Both Years [CO4, BL4]

import matplotlib.pyplot as plt
import pandas as pd

# Choose high-ranked examples with different repeated peak months.
month_name_map = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov"
}

representative_rows = []
for peak_month in [3, 10, 11]:
    options = watchlist_primary[
        watchlist_primary["Peak_Month_2010"] == peak_month
    ].sort_values(
        ["Profile_Correlation", "Repeat_Strength"],
        ascending=False,
        na_position="last"
    )
    if options.empty:
        continue
    product_key = options.index[0]
    row = options.loc[product_key]
    representative_rows.append({
        "ProductKey": product_key,
        "Description": row["Description"],
        "Repeated_Peak_Month": month_name_map[peak_month],
        "Peak_Index_2010": row["Peak_Index_2010"],
        "Peak_Index_2011": row["Peak_Index_2011"],
        "Peak_Invoices_2010": int(row["Peak_Invoices_2010"]),
        "Peak_Invoices_2011": int(row["Peak_Invoices_2011"]),
        "Profile_Correlation": row["Profile_Correlation"]
    })

selected_product_summary = pd.DataFrame(representative_rows)
print("REPRESENTATIVE PRODUCTS — MARCH, OCTOBER AND NOVEMBER PEAKS")
display(selected_product_summary)

monthly_records = []
fig, axes = plt.subplots(
    len(selected_product_summary), 2,
    figsize=(14, max(8, 3.2 * len(selected_product_summary))),
    squeeze=False
)
month_nums = list(range(1, 12))
month_labels = [month_name_map[m] for m in month_nums]

for row_idx, product in selected_product_summary.iterrows():
    product_key = product["ProductKey"]
    description = product["Description"]

    for year in [2010, 2011]:
        product_rows = work[
            (work["ProductKey"] == product_key) &
            (work["Year"] == year) &
            (work["Month"].between(1, 11))
        ]
        by_month = product_rows.groupby("Month").agg(
            Units=("Quantity", "sum"),
            Distinct_Invoices=("Invoice", "nunique")
        ).reindex(month_nums, fill_value=0)

        axes[row_idx, 0].plot(
            month_nums, by_month["Units"], marker="o", label=str(year)
        )
        axes[row_idx, 1].plot(
            month_nums, by_month["Distinct_Invoices"], marker="o", label=str(year)
        )

        for month_num in month_nums:
            monthly_records.append({
                "ProductKey": product_key,
                "Description": description,
                "Year": year,
                "Month": month_num,
                "Month_Name": month_name_map[month_num],
                "Units": int(by_month.loc[month_num, "Units"]),
                "Distinct_Invoices": int(by_month.loc[month_num, "Distinct_Invoices"])
            })

    axes[row_idx, 0].set_title(f"{description} ({product_key}) — Units")
    axes[row_idx, 1].set_title(f"{description} ({product_key}) — Invoice Activity")
    for col in range(2):
        axes[row_idx, col].set_xticks(month_nums)
        axes[row_idx, col].set_xticklabels(month_labels, rotation=45)
        axes[row_idx, col].grid(True, alpha=0.25)
        axes[row_idx, col].legend(title="Year")
    axes[row_idx, 0].set_ylabel("Units sold")
    axes[row_idx, 1].set_ylabel("Distinct invoices")

plt.suptitle(
    "Selected Product Profiles: March, October and November Peaks (2010 vs 2011)",
    fontsize=14
)
assert len(selected_product_summary) == 3, "Expected March, October and November representatives."
assert set(selected_product_summary["Repeated_Peak_Month"]) == {"Mar", "Oct", "Nov"}

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "selected_product_monthly_comparison.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

selected_product_monthly = pd.DataFrame(monthly_records)
print("Monthly quantity and invoice data prepared:", len(selected_product_monthly), "rows")
display(selected_product_monthly.head(12))

In [ ]:
# Q4: Product-Level Seasonal Demand Heatmap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Select up to three strong candidates for each peak month
ranked_candidates = watchlist_primary.sort_values(
    ["Profile_Correlation", "Repeat_Strength"],
    ascending=False,
    na_position="last"
)

selected_products = (
    ranked_candidates
    .groupby("Peak_Month_2010", sort=True)
    .head(3)
    .sort_values(
        ["Peak_Month_2010", "Profile_Correlation"],
        ascending=[True, False]
    )
)

# Calculate each product's percentage of annual-window units by month
share_2010 = u10_primary.div(
    u10_primary.sum(axis=1).replace(0, np.nan),
    axis=0
) * 100

share_2011 = u11_primary.div(
    u11_primary.sum(axis=1).replace(0, np.nan),
    axis=0
) * 100

# Average the monthly shares across the two years
heatmap_matrix = (
    share_2010.loc[selected_products.index]
    + share_2011.loc[selected_products.index]
) / 2

# Readable product labels
product_labels = [
    f"{watchlist_primary.loc[key, 'Description']} ({key})"
    for key in heatmap_matrix.index
]

month_labels = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov"
]

# Draw the visualization
fig, ax = plt.subplots(
    figsize=(14, max(7, len(heatmap_matrix) * 0.4))
)

image = ax.imshow(
    heatmap_matrix.to_numpy(),
    aspect="auto",
    cmap="YlOrRd",
    interpolation="nearest"
)

ax.set_xticks(range(11))
ax.set_xticklabels(month_labels)
ax.set_yticks(range(len(product_labels)))
ax.set_yticklabels(product_labels, fontsize=8)

ax.set_xlabel("Month")
ax.set_ylabel("Product")
ax.set_title(
    "Product-Level Seasonal Demand Heatmap\n"
    "Average share of January–November units across 2010 and 2011"
)

fig.colorbar(
    image,
    ax=ax,
    label="Average monthly share of units (%)"
)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "product_seasonality_heatmap.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

# Export the full monthly-share matrix for all 292 candidates
full_share_matrix = (
    share_2010.loc[watchlist_primary.index]
    + share_2011.loc[watchlist_primary.index]
) / 2

full_share_matrix.insert(
    0,
    "Description",
    watchlist_primary["Description"]
)

full_share_matrix.to_csv(
    OUTPUT_DIR / "full_product_month_share_matrix.csv",
    index_label="ProductKey"
)

print("Candidates in the displayed heatmap:", len(heatmap_matrix))
print("Rows in the full exported matrix:", len(full_share_matrix))
print("Files saved:")
print("-", OUTPUT_DIR / "product_seasonality_heatmap.png")
print("-", OUTPUT_DIR / "full_product_month_share_matrix.csv")

## Q5. Visualization, Sensitivity Analysis and Inventory Recommendations [CO5, BL5]

Question: What recurring patterns are visible, how sensitive is the candidate list to the screening rules, and what inventory-planning recommendations are reasonably supported by the data?

The next cells rank the candidates, visualize the peak-month distribution, and test alternative peak-strength and invoice-support thresholds. Thresholds are exploratory choices, not statistically validated cutoffs.

In [ ]:
# Q5: Peak-Month Distribution and Top 20 Candidates

import matplotlib.pyplot as plt
import pandas as pd

month_order = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov"
]

# Count candidates by their recurring peak month
peak_counts = (
    watchlist_primary["Peak_Month_Name"]
    .value_counts()
    .reindex(month_order, fill_value=0)
)

print("RECURRING-PEAK CANDIDATES BY MONTH")
print(peak_counts.to_string())
print("\nTotal candidates:", int(peak_counts.sum()))

assert int(peak_counts.sum()) == len(watchlist_primary), \
    "Peak-month counts do not match the watchlist!"

# Create bar chart
fig, ax = plt.subplots(figsize=(11, 5))

bars = ax.bar(
    peak_counts.index,
    peak_counts.values
)

ax.bar_label(bars, padding=3)
ax.set_title("Distribution of Recurring Product-Demand Peaks")
ax.set_xlabel("Peak month")
ax.set_ylabel("Number of candidate products")
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "peak_month_distribution.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

# Prepare the top 20 candidates
top20_primary = watchlist_primary.sort_values(
    ["Profile_Correlation", "Repeat_Strength"],
    ascending=False,
    na_position="last"
).head(20)

top20_table = top20_primary[[
    "Description",
    "Peak_Month_Name",
    "Peak_Index_2010",
    "Peak_Index_2011",
    "Peak_Invoices_2010",
    "Peak_Invoices_2011",
    "Profile_Correlation"
]].copy()

top20_table.index.name = "ProductKey"

print("\nTOP 20 RECURRING-PEAK CANDIDATES")
display(top20_table)

print("\nBar chart saved to", OUTPUT_DIR / "peak_month_distribution.png")

In [ ]:
# Q5: Sensitivity Analysis of Product Screening Rules

import pandas as pd

# Test different peak-to-average thresholds
peak_thresholds = [1.25, 1.5, 2.0]
results = []

for threshold in peak_thresholds:

    # Rule 1: Same peak month and sufficient peak strength
    base_candidates = eligible_primary[
        eligible_primary["Same_Peak_Month"] &
        (eligible_primary["Peak_Index_2010"] >= threshold) &
        (eligible_primary["Peak_Index_2011"] >= threshold)
    ].copy()

    # Rule 2: Also require >= 10 distinct invoices at each peak
    supported_candidates = base_candidates[
        (base_candidates["Peak_Invoices_2010"] >= 10) &
        (base_candidates["Peak_Invoices_2011"] >= 10)
    ]

    results.append({
        "Peak Index Threshold": threshold,
        "Same-Peak Candidates": len(base_candidates),
        "Candidates With Peak Invoice Support": len(supported_candidates)
    })

sensitivity_results = pd.DataFrame(results)

print("SENSITIVITY TO PEAK-STRENGTH THRESHOLD")
display(sensitivity_results)

# Separately test the minimum invoice requirement,
# keeping the peak-index threshold fixed at 1.5
invoice_results = []

base = eligible_primary[
    eligible_primary["Same_Peak_Month"] &
    (eligible_primary["Peak_Index_2010"] >= 1.5) &
    (eligible_primary["Peak_Index_2011"] >= 1.5)
].copy()

for minimum_invoices in [0, 10, 20]:

    if minimum_invoices == 0:
        selected = base
    else:
        selected = base[
            (base["Peak_Invoices_2010"] >= minimum_invoices) &
            (base["Peak_Invoices_2011"] >= minimum_invoices)
        ]

    invoice_results.append({
        "Minimum Invoices At Peak (each year)": minimum_invoices,
        "Candidate Count": len(selected)
    })

invoice_sensitivity = pd.DataFrame(invoice_results)

print("\nSENSITIVITY TO PEAK-MONTH INVOICE REQUIREMENT")
display(invoice_sensitivity)

### Interpretation and Limitations

In the current primary analysis, 292 products meet the screening rules, and 156 have a November peak (53.4% of the candidate list). October has 33 candidates and March has 28. These counts describe the screened candidate list, not the proportion of all retail products that are seasonal.

The dataset covers only two comparable annual windows. Invoice counts measure transaction activity, not unique customers. The analysis retains zero-price records, excludes non-positive quantities and cancellation-coded invoices, and therefore describes retained positive-quantity transactions rather than net sales after returns. Similar descriptions are used to consolidate stock-code case variants only under a documented rule; descriptions alone do not guarantee true product identity.

In [ ]:
# FINAL STEP: Validate and Export the EDAV Project

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

output_dir = OUTPUT_DIR / "EDAV_Final_Outputs"
os.makedirs(output_dir, exist_ok=True)

# --------------------------------------------------
# 1. VALIDATE THE FINAL PRODUCT WATCHLIST
# --------------------------------------------------

final_watchlist = watchlist_primary.copy()

assert len(final_watchlist) == 292, \
    "Candidate count is not 292. Review the primary analysis."

assert (final_watchlist["Peak_Month_2010"] ==
        final_watchlist["Peak_Month_2011"]).all(), \
    "Some candidates do not repeat their peak month."

assert (final_watchlist["Peak_Index_2010"] >= 1.5).all()
assert (final_watchlist["Peak_Index_2011"] >= 1.5).all()

assert (final_watchlist["Peak_Invoices_2010"] >= 10).all()
assert (final_watchlist["Peak_Invoices_2011"] >= 10).all()

assert (final_watchlist["Invoices_2010"] >= 10).all()
assert (final_watchlist["Invoices_2011"] >= 10).all()

print("Candidate validation passed:", len(final_watchlist))

# Check the comparison window
assert set(primary_data["Year"].unique()) == {2010, 2011}
assert primary_data["Month"].between(1, 11).all()

print("Comparison window validated: January-November 2010 vs 2011.")

# --------------------------------------------------
# 2. EXPORT THE CLEANING SUMMARY
# --------------------------------------------------

cleaning_summary.to_csv(
    f"{output_dir}/data_cleaning_summary.csv",
    index=False
)

# Missing values before/after, restricted to original columns
original_columns = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country"
]

missing_audit = pd.DataFrame({
    "Missing_Before": df[original_columns].isna().sum(),
    "Missing_After": df_clean[original_columns].isna().sum()
})

missing_audit.to_csv(
    f"{output_dir}/missing_values_audit.csv"
)

# --------------------------------------------------
# 3. EXPORT THE FINAL WATCHLIST AND TOP 20
# --------------------------------------------------

final_watchlist.to_csv(
    f"{output_dir}/final_seasonal_product_watchlist.csv",
    index_label="ProductKey"
)

top20 = final_watchlist.head(20)

top20.to_csv(
    f"{output_dir}/top_20_seasonal_products.csv",
    index_label="ProductKey"
)

# --------------------------------------------------
# 4. EXPORT MONTHLY QUANTITY, REVENUE AND INVOICES
# --------------------------------------------------

monthly_final = (
    primary_data.groupby(["Year", "Month"])
    .agg(
        Total_Units=("Quantity", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Distinct_Invoices=("Invoice", "nunique")
    )
    .reset_index()
)

monthly_final.to_csv(
    f"{output_dir}/primary_monthly_demand_summary.csv",
    index=False
)

# Peak-month distribution directly from the final watchlist
month_order = [
    "Jan", "Feb", "Mar", "Apr", "May",
    "Jun", "Jul", "Aug", "Sep", "Oct", "Nov"
]

peak_distribution = (
    final_watchlist["Peak_Month_Name"]
    .value_counts()
    .reindex(month_order, fill_value=0)
    .rename_axis("Peak_Month")
    .reset_index(name="Candidate_Count")
)

assert peak_distribution["Candidate_Count"].sum() == 292

assert int(
    peak_distribution.loc[
        peak_distribution["Peak_Month"] == "Nov",
        "Candidate_Count"
    ].iloc[0]
) == 156

peak_distribution.to_csv(
    f"{output_dir}/peak_month_distribution.csv",
    index=False
)

# Peak-month bar chart
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(
    peak_distribution["Peak_Month"],
    peak_distribution["Candidate_Count"]
)

ax.bar_label(bars, padding=3)
ax.set_title("Recurring Product-Demand Candidates by Peak Month")
ax.set_xlabel("Peak Month")
ax.set_ylabel("Number of Candidates")
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(
    f"{output_dir}/peak_month_distribution.png",
    dpi=200,
    bbox_inches="tight"
)
plt.close(fig)

# --------------------------------------------------
# 5. SAVE THE QUANTITY AND REVENUE HEATMAPS
# --------------------------------------------------

units = monthly_final.pivot(
    index="Year", columns="Month", values="Total_Units"
).reindex(index=[2010, 2011], columns=range(1, 12))

revenue = monthly_final.pivot(
    index="Year", columns="Month", values="Total_Revenue"
).reindex(index=[2010, 2011], columns=range(1, 12))

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

for ax, table, title, color_map, scale in [
    (axes[0], units, "Monthly Total Units Sold", "YlOrRd", 1),
    (axes[1], revenue / 1000, "Monthly Total Revenue (thousands)", "YlGnBu", 1000)
]:
    image = ax.imshow(table.to_numpy(), aspect="auto", cmap=color_map)
    ax.set_title(title)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["2010", "2011"])
    ax.set_xticks(range(11))
    ax.set_xticklabels(month_order)
    ax.set_xlabel("Month")

    for i in range(2):
        for j in range(11):
            ax.text(
                j, i, f"{table.iloc[i, j]:,.0f}",
                ha="center", va="center", fontsize=8
            )

    fig.colorbar(image, ax=ax)

fig.suptitle("Retail Demand Comparison: January-November 2010 vs 2011")
plt.tight_layout()

plt.savefig(
    f"{output_dir}/monthly_quantity_revenue_heatmaps.png",
    dpi=200,
    bbox_inches="tight"
)
plt.close(fig)

# --------------------------------------------------
# 6. EXPORT THE FULL PRODUCT-MONTH HEATMAP MATRIX
# --------------------------------------------------

share_2010 = u10_primary.div(
    u10_primary.sum(axis=1).replace(0, np.nan), axis=0
) * 100

share_2011 = u11_primary.div(
    u11_primary.sum(axis=1).replace(0, np.nan), axis=0
) * 100

full_matrix = (
    share_2010.loc[final_watchlist.index]
    + share_2011.loc[final_watchlist.index]
) / 2

full_matrix.insert(
    0, "Description", final_watchlist["Description"]
)

full_matrix.to_csv(
    f"{output_dir}/full_product_month_share_matrix.csv",
    index_label="ProductKey"
)

# Preserve the selected-product heatmap from our earlier cell
source_heatmap = OUTPUT_DIR / "product_seasonality_heatmap.png"

if os.path.exists(source_heatmap):
    import shutil
    shutil.copy2(
        source_heatmap,
        f"{output_dir}/product_seasonality_heatmap.png"
    )

# --------------------------------------------------
# 7. EXPORT SENSITIVITY RESULTS
# --------------------------------------------------

sensitivity_results.to_csv(
    f"{output_dir}/peak_threshold_sensitivity.csv",
    index=False
)

invoice_sensitivity.to_csv(
    f"{output_dir}/invoice_threshold_sensitivity.csv",
    index=False
)

# --------------------------------------------------
# 7. Export the selected-product evidence

selected_product_summary.to_csv(
    f"{output_dir}/representative_products_summary.csv",
    index=False
)
selected_product_monthly.to_csv(
    f"{output_dir}/representative_product_monthly_comparison.csv",
    index=False
)

comparison_chart = OUTPUT_DIR / "selected_product_monthly_comparison.png"
if os.path.exists(comparison_chart):
    import shutil
    shutil.copy2(
        comparison_chart,
        f"{output_dir}/selected_product_monthly_comparison.png"
    )
else:
    raise FileNotFoundError(
        "Run the representative-product comparison cell before final export."
    )

# Save the monthly line charts generated under Q2
import shutil
for source_chart, output_name in [
    (OUTPUT_DIR / "monthly_product_demand_trend.png", "monthly_product_demand_trend.png"),
    (OUTPUT_DIR / "monthly_demand_year_over_year.png", "monthly_demand_year_over_year.png")
]:
    if os.path.exists(source_chart):
        shutil.copy2(source_chart, f"{output_dir}/{output_name}")
    else:
        raise FileNotFoundError(
            f"Run the monthly trend chart cell before final export: {source_chart}"
        )

# 8. DOCUMENT THE METHOD AND LIMITATIONS
# --------------------------------------------------

with open(
    f"{output_dir}/methodology_and_limitations.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(
        "SEASONAL PRODUCT DEMAND ANALYSIS\n\n"
        "Primary comparison: January-November 2010 versus "
        "January-November 2011.\n\n"
        "The candidate screening rules require at least 10 distinct "
        "invoices in each year, the same peak month in both years, "
        "a peak-to-average monthly quantity ratio of at least 1.5 "
        "in each year, and at least 10 distinct invoices during "
        "each peak month.\n\n"
        "The analysis identified 292 recurring-peak candidates from "
        "2,358 eligible products. November was the most frequent peak "
        "month, with 156 candidates.\n\n"
        "These are exploratory recurring-peak candidates, not "
        "statistically proven long-term seasonal products. The data "
        "contains only two comparable annual windows.\n\n"
        "The primary analysis excludes non-positive quantities and "
        "cancellation-coded invoices, so it describes retained "
        "positive-quantity transactions rather than net sales after "
        "returns. Zero-price rows were retained and counted. Missing "
        "customer identifiers were allowed because customer-level "
        "identification is not required for product-level analysis.\n"
    )

# --------------------------------------------------
# 9. FINAL VALIDATION AND ZIP EXPORT
# --------------------------------------------------

assert len(final_watchlist) == 292
assert int(peak_distribution["Candidate_Count"].sum()) == 292

zip_path = OUTPUT_DIR / "EDAV_Final_Outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for filename in sorted(os.listdir(output_dir)):
        archive.write(
            os.path.join(output_dir, filename),
            arcname=filename
        )

print("\nFINAL VALIDATION PASSED")
print("Eligible products:", len(eligible_primary))
print("Recurring-peak candidates:", len(final_watchlist))
print("November peaks:", int(
    peak_distribution.loc[
        peak_distribution["Peak_Month"] == "Nov",
        "Candidate_Count"
    ].iloc[0]
))

print("\nFILES INCLUDED IN FINAL ZIP:")
for filename in sorted(os.listdir(output_dir)):
    print("-", filename)

if IN_COLAB:
    print("\nDownloading:", zip_path)
    colab_files.download(str(zip_path))
else:
    print("\nFinal ZIP saved at:", zip_path.resolve())

## Final Output Checklist

The last cell verifies the analysis period, screening criteria, candidate count and November count, then exports the report-ready tables, heatmaps, sensitivity tables and methodology notes into `EDAV_Final_Outputs.zip`. Keep the ZIP and the downloaded `.ipynb` together for project submission.